# TrialMatch — 임상시험 연결 검색 (Gemini + ClinicalTrials.gov)

치료 선택지가 소진된 환자가 '자기 상황'을 검색창에 적으면 → **현재 모집 중인 실제 임상시험**을
ClinicalTrials.gov 에서 조회하고, Gemini 가 선정기준과 대조해 적합도를 매겨 연결한다.

**흐름**: ⓪초기화 → ①설치 → ②파일 기록 → ③Gemini 키 입력 → ④서버 실행(cloudflared URL).

> ⚠️ 임상시험 정보는 공개 등록 데이터(모집중)만 조회합니다. 적합도 점수는 Gemini의 참고
> 추정치이며, 최종 참여 적격 여부·연락은 각 실시기관의 절차를 따릅니다. 본 도구는 환자를
> 직접 모집·중개하지 않습니다.

> 🔴 **다른 앱(OncoReg)과 안 섞이게** — 이 앱은 전용 폴더(`trialmatch/`), 전용 모듈명
> (`trialmatch_app`), 전용 포트(8001)를 씁니다. 그래도 한 런타임에서 여러 앱을 돌렸다면
> **런타임 → 세션 다시 시작** 후 이 노트북만 실행하는 게 가장 확실합니다.


## 0) 초기화 — 이전에 돌린 다른 앱의 캐시/포트 정리

In [ ]:
import sys
for _m in ['server', 'oncoreg_app', 'trialmatch_app']:
    sys.modules.pop(_m, None)
print('모듈 캐시 정리 완료. (포트가 이미 사용 중이면 런타임 다시 시작을 권장)')


## 1) 패키지 설치

In [ ]:
!pip -q install google-genai flask flask-cors flask-cloudflared pydantic


## 2) 백엔드/프론트 파일 기록 (전용 폴더 `trialmatch/`)

In [ ]:
import os; os.makedirs('trialmatch/templates', exist_ok=True); print('trialmatch/ 준비 완료')


In [ ]:
%%writefile trialmatch/trialmatch_app.py
# © 2026 권도융 (Kwon Do-yung). All Rights Reserved. 무단 복제·사용·배포 금지 — /LICENSE 참조.
"""
TrialMatch — 임상시험 연결 검색 플랫폼 (프로토타입)

시나리오: 표준치료·희귀의약품으로도 치료가 어려운 환자가 '자기 상황'을 자유롭게
검색창에 적으면 → 현재 '모집 중'인 임상시험을 찾아 적합도를 대조하고 연락처로 연결한다.

설계 원칙 (윤리/규제 경계 = "IRB/규제 승인된 부분만")
  - 임상시험 정보는 우리가 지어내지 않는다. **ClinicalTrials.gov 공개 API(v2)** 의
    '실제 등록·모집중 시험'만 가져온다. (중재연구는 IRB 승인 하에 등록됨)
  - Gemini 는 (1) 한글 자유입력을 검색어로 해석하고 (2) 각 시험의 선정/제외 기준과
    환자를 대조해 적합도를 매기는 '보조' 역할만 한다. 판단·연결의 주체가 아니다.
  - 본 도구는 환자를 직접 모집·중개하지 않는다. 표시되는 연락처는 각 시험의 '공식
    등록 연락처'이며, 최종 참여 결정과 적격 판정은 실시기관과 담당 의료진이 한다.

역할
  - /            : templates/index.html 서빙
  - /api/health  : 상태
  - /api/search  : 자유입력 → (Gemini 해석) → CT.gov 조회 → (Gemini 적합도 채점) → 결과

로컬:  GEMINI_API_KEY=... python server.py   ->  http://localhost:8000
Colab: TrialMatch_Colab.ipynb 참고 (cloudflared 터널)
"""
import os
import json
import urllib.parse
import urllib.request
from typing import List, Optional

from flask import Flask, request, jsonify, render_template
from flask_cors import CORS

from google import genai
from google.genai import types
from pydantic import BaseModel

DEFAULT_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
CTGOV_API = "https://clinicaltrials.gov/api/v2/studies"
# '진행 중'으로 볼 상태 (모집중 + 모집예정). 필요시 ACTIVE_NOT_RECRUITING 추가 가능.
OPEN_STATUSES = ["RECRUITING", "NOT_YET_RECRUITING"]
MAX_TRIALS_SCORE = 8   # 적합도 채점에 넘길 최대 시험 수(토큰/속도 관리)


# ----------------------------------------------------------------------------
# 스키마
# ----------------------------------------------------------------------------
class ExtractedQuery(BaseModel):
    condition_en: str        # CT.gov 검색용 '영어' 질환명 (한글 입력을 번역/정규화)
    other_terms: str = ""    # 개입/약물/바이오마커 등 추가 영어 키워드(공백 구분)
    location_en: str = ""    # 지역(국가/도시) 영어, 없으면 ""
    summary_ko: str          # 사용자가 확인할 한글 요약


class Criterion(BaseModel):
    status: str              # y(부합) | n(불충족) | q(확인 필요)
    text: str                # 기준 문장(한국어)


class TrialMatch(BaseModel):
    idx: int                 # 입력 시험 목록에서의 순번(0부터)
    matchPct: int            # 적합도 0~100
    summary: str             # 이 환자 기준 한 줄 한글 요약
    crit: List[Criterion]    # 핵심 선정/제외 기준 대조 3~6개
    note: str                # 점수 근거 짧게(한국어)


# ----------------------------------------------------------------------------
# Gemini
# ----------------------------------------------------------------------------
def get_client() -> genai.Client:
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not key:
        raise RuntimeError("GEMINI_API_KEY 환경변수가 설정되지 않았습니다.")
    return genai.Client(api_key=key)


def gemini_json(prompt: str, schema, temperature=0.3):
    resp = get_client().models.generate_content(
        model=DEFAULT_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,
            temperature=temperature,
        ),
    )
    parsed = getattr(resp, "parsed", None)
    if parsed is not None:
        if isinstance(parsed, list):
            return [p.model_dump() for p in parsed]
        return parsed.model_dump()
    return json.loads(resp.text)


def extract_query(text: str) -> dict:
    prompt = f"""환자/보호자가 자유롭게 적은 아래 문장에서, 임상시험 데이터베이스
(ClinicalTrials.gov, 영어) 검색에 쓸 핵심어를 뽑아라. 반드시 JSON 스키마를 따른다.

[입력]
{text}

[규칙]
- condition_en: 핵심 '질환명'을 영어로. (예: 한글 "전신성 경화증 폐동맥고혈압" ->
  "systemic sclerosis pulmonary arterial hypertension"). 가장 특이적인 진단명 위주.
- other_terms: 약물/바이오마커/이전치료/변이 등 추가 영어 키워드를 공백으로. 없으면 "".
- location_en: 지역이 드러나면 영어 국가/도시(예: "Korea","Seoul"), 없으면 "".
- summary_ko: 입력을 1~2문장 한글로 요약(사용자 확인용)."""
    try:
        return gemini_json(prompt, ExtractedQuery)
    except Exception:
        # Gemini 실패 시: 입력을 그대로 검색어로(영문 입력이면 그대로 먹힘)
        return {"condition_en": text[:120], "other_terms": "", "location_en": "",
                "summary_ko": text[:200]}


def score_trials(text: str, trials: List[dict]) -> List[dict]:
    slim = []
    for i, t in enumerate(trials):
        slim.append({
            "idx": i,
            "title": t.get("title"),
            "conditions": t.get("conditions"),
            "phases": t.get("phases"),
            "sex": t.get("sex"), "minAge": t.get("minAge"), "maxAge": t.get("maxAge"),
            "eligibility": (t.get("eligibilityText") or "")[:1200],
        })
    prompt = f"""아래는 한 환자의 상황과, 현재 모집 중인 실제 임상시험 목록이다.
각 시험의 선정/제외 기준(eligibility)을 환자와 대조해 적합도를 매겨라.
반드시 각 시험(idx)마다 하나씩, JSON 배열로 답한다.

[환자 상황]
{text}

[임상시험 목록(JSON)]
{json.dumps(slim, ensure_ascii=False)}

[규칙]
- idx: 위 목록의 순번 그대로.
- matchPct: 환자 조건이 이 시험 선정기준에 얼마나 부합하는지 0~100 정수.
- crit: 이 시험의 '핵심' 선정/제외 기준 3~6개를 골라 한국어로 요약하고, 환자 기준
  status 를 y(부합)/n(불충족)/q(정보부족·확인필요)로 표기.
- summary: 이 환자에게 이 시험이 왜 후보인지 한 줄 한국어.
- note: 점수 근거를 한 문장 한국어로.
- 정보가 부족하면 단정하지 말고 q 로. 과장 금지."""
    try:
        return gemini_json(prompt, list[TrialMatch])
    except Exception:
        return []


# ----------------------------------------------------------------------------
# ClinicalTrials.gov (실데이터)
# ----------------------------------------------------------------------------
def fetch_ctgov(q: dict, page_size=15) -> List[dict]:
    params = {
        "query.cond": q.get("condition_en", "") or "",
        "filter.overallStatus": ",".join(OPEN_STATUSES),
        "pageSize": str(page_size),
        "format": "json",
    }
    if q.get("other_terms"):
        params["query.term"] = q["other_terms"]
    if q.get("location_en"):
        params["query.locn"] = q["location_en"]

    def _req(p):
        url = CTGOV_API + "?" + urllib.parse.urlencode(p)
        req = urllib.request.Request(url, headers={"User-Agent": "TrialMatch-Prototype/1.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            return json.loads(r.read().decode("utf-8"))

    try:
        data = _req(params)
    except Exception:
        # 상태 필터가 문제일 수 있으니 한 번 더 시도(필터 제거)
        p2 = {k: v for k, v in params.items() if k != "filter.overallStatus"}
        try:
            data = _req(p2)
        except Exception:
            return []

    out = []
    for study in data.get("studies", []):
        ps = study.get("protocolSection", {})
        idm = ps.get("identificationModule", {})
        stm = ps.get("statusModule", {})
        status = stm.get("overallStatus", "")
        if status and status not in OPEN_STATUSES:
            continue  # 혹시 필터 없이 온 경우 후처리
        elig = ps.get("eligibilityModule", {})
        cl = ps.get("contactsLocationsModule", {})
        locs = []
        for l in (cl.get("locations", []) or [])[:6]:
            locs.append({
                "facility": l.get("facility"), "city": l.get("city"),
                "country": l.get("country"), "status": l.get("status"),
            })
        contacts = []
        for cc in (cl.get("centralContacts", []) or [])[:3]:
            contacts.append({"name": cc.get("name"), "phone": cc.get("phone"),
                             "email": cc.get("email")})
        nct = idm.get("nctId", "")
        out.append({
            "nctId": nct,
            "title": idm.get("briefTitle") or idm.get("officialTitle") or nct,
            "status": status,
            "phases": ps.get("designModule", {}).get("phases", []) or [],
            "conditions": ps.get("conditionsModule", {}).get("conditions", []) or [],
            "interventions": [i.get("name") for i in
                              ps.get("armsInterventionsModule", {}).get("interventions", []) or []
                              if i.get("name")],
            "eligibilityText": elig.get("eligibilityCriteria", "") or "",
            "sex": elig.get("sex"), "minAge": elig.get("minimumAge"),
            "maxAge": elig.get("maximumAge"),
            "locations": locs, "contacts": contacts,
            "url": f"https://clinicaltrials.gov/study/{nct}" if nct else "",
        })
    return out


# 실 API 가 막힌 환경을 위한 '표본' 데이터 (명확히 데모 표시). 구조는 실데이터와 동일.
SAMPLE_TRIALS = [
    {"nctId": "NCT00000000",
     "title": "[표본] 결합조직질환 관련 폐동맥고혈압 대상 신규 경구제 3상",
     "status": "RECRUITING", "phases": ["PHASE3"],
     "conditions": ["Pulmonary Arterial Hypertension", "Systemic Sclerosis"],
     "interventions": ["Investigational oral agent"],
     "eligibilityText": "Inclusion: adults with CTD-associated PAH, WHO FC II-III, 6MWD 150-450m, "
                        "on stable background therapy >=3 months. Exclusion: PVOD, severe hepatic impairment.",
     "sex": "ALL", "minAge": "19 Years", "maxAge": "80 Years",
     "locations": [{"facility": "[표본] 서울 소재 대학병원", "city": "Seoul", "country": "Korea, Republic of", "status": "RECRUITING"}],
     "contacts": [{"name": "[표본] 임상시험 코디네이터", "phone": "02-000-0000", "email": "trial@example.org"}],
     "url": "https://clinicaltrials.gov/"},
    {"nctId": "NCT00000001",
     "title": "[표본] 희귀 고형암 환자 대상 표적치료제 바스켓 2상",
     "status": "RECRUITING", "phases": ["PHASE2"],
     "conditions": ["Rare Solid Tumor"],
     "interventions": ["Targeted agent (biomarker-selected)"],
     "eligibilityText": "Inclusion: advanced rare solid tumor with actionable alteration, progressed on standard therapy, "
                        "ECOG 0-1. Exclusion: untreated CNS metastases.",
     "sex": "ALL", "minAge": "18 Years", "maxAge": "N/A",
     "locations": [{"facility": "[표본] 국내 다기관", "city": "Seoul", "country": "Korea, Republic of", "status": "RECRUITING"}],
     "contacts": [{"name": "[표본] 연구간호사", "phone": "02-000-0001", "email": "basket@example.org"}],
     "url": "https://clinicaltrials.gov/"},
]


def normalize_crit(c):
    if c.get("status") not in ("y", "n", "q"):
        c["status"] = "q"
    return c


# ----------------------------------------------------------------------------
# Flask
# ----------------------------------------------------------------------------
def create_app() -> Flask:
    app = Flask(__name__, template_folder="templates")
    CORS(app)

    @app.get("/")
    def index():
        return render_template("index.html")

    @app.get("/api/health")
    def health():
        return jsonify({
            "ok": True, "model": DEFAULT_MODEL,
            "key_present": bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")),
        })

    @app.post("/api/search")
    def search():
        body = request.get_json(force=True) or {}
        text = (body.get("text") or "").strip()
        if not text:
            return jsonify({"error": "검색어(환자 상황)를 입력하세요."}), 400
        try:
            q = extract_query(text)
            trials = fetch_ctgov(q)
            source = "clinicaltrials.gov"
            if not trials:
                trials = SAMPLE_TRIALS
                source = "sample"     # 실 API 미도달 → 표본 데이터
            top = trials[:MAX_TRIALS_SCORE]
            matches = {m["idx"]: m for m in score_trials(text, top)}
            results = []
            for i, t in enumerate(top):
                m = matches.get(i, {})
                results.append({
                    **t,
                    "matchPct": max(0, min(100, int(m.get("matchPct", 0)))) if m else None,
                    "summary": m.get("summary", ""),
                    "note": m.get("note", ""),
                    "crit": [normalize_crit(c) for c in m.get("crit", [])],
                })
            results.sort(key=lambda r: (r["matchPct"] is None, -(r["matchPct"] or 0)))
            return jsonify({"query": q, "source": source, "count": len(results), "results": results})
        except Exception as e:
            return jsonify({"error": str(e)}), 502

    return app


if __name__ == "__main__":
    port = int(os.environ.get("PORT", "8000"))
    print(f"[TrialMatch] http://localhost:{port}  (model={DEFAULT_MODEL})")
    create_app().run(host="0.0.0.0", port=port, debug=False)


In [ ]:
%%writefile trialmatch/templates/index.html
<!DOCTYPE html>
<!-- © 2026 권도융 (Kwon Do-yung). All Rights Reserved. 무단 복제·사용·배포 금지 — /LICENSE 참조. -->
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>TrialMatch — 임상시험 연결 검색 (Gemini + ClinicalTrials.gov)</title>
<style>
:root{
  --paper:#FBFAF7;--card:#FFF;--ink:#16181A;--muted:#6B6E73;--faint:#9A9C9F;
  --rule:#E3DFD7;--rule2:#EFECE6;
  --seal:#0E6E5E;--seal-bg:#E8F2EF;
  --alert:#B03A2E;--alert-bg:#FBEEEC;
  --trial:#5A4B8C;--trial-bg:#EEEBF5;
  --amber:#9A6A00;--amber-bg:#FBF3DF;
  --mono:ui-monospace,"SF Mono",Menlo,Consolas,"D2Coding",monospace;
  --sans:-apple-system,BlinkMacSystemFont,"Pretendard","Malgun Gothic","Noto Sans KR",sans-serif;
}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--paper);color:var(--ink);font-family:var(--sans);font-size:15px;line-height:1.65;-webkit-font-smoothing:antialiased}
.wrap{max-width:1120px;margin:0 auto;padding:0 28px 80px}
.demo{background:var(--alert-bg);border-bottom:1px solid #E9CFC9;padding:9px 28px;font-size:12.5px;color:#8A2F26;text-align:center}
.demo b{font-weight:600}
header{padding:30px 0 20px;border-bottom:1px solid var(--rule);margin-bottom:22px}
.brand{display:flex;align-items:baseline;gap:11px;flex-wrap:wrap}
.brand h1{font-size:20px;font-weight:600;letter-spacing:-.02em}
.brand .tag{font-size:12px;color:var(--muted);border-left:1px solid var(--rule);padding-left:11px}
.brand .eng{font-size:11px;color:#fff;background:var(--seal);padding:2px 8px;border-radius:10px;font-family:var(--mono)}
.brand .eng.off{background:var(--faint)}
.irb{background:var(--amber-bg);border:1px solid #EAD9A8;border-radius:3px;padding:11px 15px;font-size:12px;color:#6E4E00;margin:16px 0 22px;line-height:1.6}
.irb b{font-weight:600}
.searchcard{background:var(--card);border:1px solid var(--rule);border-radius:4px;padding:20px}
.searchcard label{display:block;font-size:12.5px;color:var(--muted);margin-bottom:7px}
.searchcard textarea{width:100%;min-height:82px;resize:vertical;padding:12px 13px;border:1px solid var(--rule);border-radius:3px;font-family:inherit;font-size:14.5px;line-height:1.6;color:var(--ink);background:#fff}
.searchcard textarea:focus{outline:2px solid var(--seal);outline-offset:-1px;border-color:transparent}
.chips{display:flex;gap:8px;flex-wrap:wrap;margin-top:11px}
.chip{font-size:12px;color:var(--muted);background:#F4F1EB;border:1px solid var(--rule2);border-radius:14px;padding:5px 12px;cursor:pointer}
.chip:hover{border-color:var(--seal);color:var(--seal)}
.searchrow{margin-top:15px;display:flex;gap:14px;align-items:center;flex-wrap:wrap}
button.go{background:var(--ink);color:#fff;border:0;border-radius:3px;padding:11px 24px;font-family:inherit;font-size:14px;font-weight:500;cursor:pointer}
button.go:hover{background:#2C2F33} button.go:disabled{background:#C9C6C0;cursor:not-allowed}
.chk{font-size:12.5px;color:var(--muted);display:flex;align-items:center;gap:6px;cursor:pointer}
.qbar{margin:22px 0 14px;display:flex;justify-content:space-between;align-items:flex-start;gap:14px;flex-wrap:wrap}
.qbar .qsum{font-size:13px;color:var(--ink)}
.qbar .qsum b{color:var(--muted);font-weight:500}
.qbar .qterms{font-size:11.5px;color:var(--faint);font-family:var(--mono);margin-top:3px}
.srcbadge{font-size:11px;font-family:var(--mono);padding:3px 9px;border-radius:2px;white-space:nowrap}
.srcbadge.live{background:var(--seal-bg);color:var(--seal)}
.srcbadge.sample{background:var(--amber-bg);color:var(--amber)}
.split{display:grid;grid-template-columns:1fr 1fr;gap:20px}
@media(max-width:900px){.split{grid-template-columns:1fr}}
.tcard{border:1px solid var(--rule);border-radius:4px;background:#fff;margin-bottom:11px;overflow:hidden;cursor:pointer}
.tcard:hover{border-color:#C9C4BA}
.tcard.act{border-color:var(--seal)}
.tcard .thead{padding:14px 16px}
.tstatus{display:inline-block;font-size:11px;font-weight:600;padding:2px 8px;border-radius:2px;margin-bottom:7px}
.tstatus.rec{background:var(--seal-bg);color:var(--seal)}
.tstatus.soon{background:var(--trial-bg);color:var(--trial)}
.tcard .ttl{font-size:14.5px;font-weight:500;line-height:1.45}
.tcard .meta{font-size:11.5px;color:var(--muted);font-family:var(--mono);margin-top:5px}
.tcard .conds{margin-top:7px;display:flex;flex-wrap:wrap;gap:5px}
.tcard .cond{font-size:11px;background:#F2EFE9;color:var(--muted);border-radius:2px;padding:1px 7px}
.tcard .sum{font-size:12.5px;color:#33363A;margin-top:9px;line-height:1.55}
.matchbar{display:flex;align-items:center;gap:9px;margin-top:11px}
.matchtrack{flex:1;height:6px;background:#EEEBE5;border-radius:3px;overflow:hidden}
.matchfill{height:100%;background:var(--seal)}
.matchpct{font-family:var(--mono);font-size:13px;font-weight:600;color:var(--seal)}
.matchpct.none{color:var(--faint)}
.tcard .foot{padding:8px 16px;border-top:1px solid var(--rule2);background:#FCFBF9;font-size:11.5px;color:var(--muted);display:flex;justify-content:space-between;align-items:center;gap:10px}
.viewer{position:sticky;top:20px}
.vempty{border:1px dashed var(--rule);border-radius:4px;padding:44px 24px;text-align:center;color:var(--faint);font-size:13px;background:#FCFBF9}
.vbox{border:1px solid var(--rule);border-radius:4px;background:#fff;overflow:hidden}
.vhead{padding:13px 16px;border-bottom:1px solid var(--rule);background:#FCFBF9}
.vhead .vt{font-size:13.5px;font-weight:500;line-height:1.4}
.vhead .vm{font-size:11px;color:var(--muted);font-family:var(--mono);margin-top:3px}
.vsec{padding:14px 16px;border-bottom:1px solid var(--rule2)}
.vsec:last-child{border-bottom:0}
.vsec .vl{font-size:11px;color:var(--faint);text-transform:uppercase;letter-spacing:.04em;margin-bottom:8px}
.crit{list-style:none}
.crit li{font-size:12.5px;padding:3px 0 3px 20px;position:relative;line-height:1.55}
.crit li.y::before{content:"✓";position:absolute;left:0;color:var(--seal);font-weight:600}
.crit li.n::before{content:"—";position:absolute;left:0;color:var(--alert)}
.crit li.q::before{content:"?";position:absolute;left:0;color:var(--faint);font-weight:600}
.crit li.n{color:var(--alert)}
.eltext{font-size:12px;line-height:1.7;color:#44474B;white-space:pre-wrap;max-height:220px;overflow:auto;background:#FCFBF9;border:1px solid var(--rule2);border-radius:3px;padding:11px}
.loc{font-size:12.5px;color:#33363A;padding:5px 0;border-bottom:1px solid var(--rule2)}
.loc:last-child{border-bottom:0}
.loc .lc{font-size:11px;color:var(--faint);font-family:var(--mono)}
.contact{font-size:12.5px;color:#33363A;font-family:var(--mono);padding:2px 0}
.inq{font-size:12.5px;line-height:1.7;color:#33363A;white-space:pre-wrap;background:var(--seal-bg);border:1px solid #C5DFD8;border-radius:3px;padding:12px}
.btnrow{margin-top:10px;display:flex;gap:8px;flex-wrap:wrap}
button.ghost{background:#fff;color:var(--ink);border:1px solid var(--rule);border-radius:3px;padding:7px 13px;font-family:inherit;font-size:12.5px;cursor:pointer}
button.ghost:hover{background:#FAF8F4}
a.ext{font-size:12px;color:var(--seal);text-decoration:none;font-family:var(--mono)}
a.ext:hover{text-decoration:underline}
.note{font-size:11.5px;color:var(--faint);margin-top:10px;line-height:1.6}
.spin{display:inline-block;width:12px;height:12px;border:2px solid var(--rule);border-top-color:var(--ink);border-radius:50%;animation:sp .7s linear infinite;vertical-align:-2px;margin-right:7px}
@keyframes sp{to{transform:rotate(360deg)}}
@media(prefers-reduced-motion:reduce){.spin{animation:none}}
.empty{border:1px dashed var(--rule);border-radius:4px;padding:40px 24px;text-align:center;color:var(--faint);font-size:13px;background:#FCFBF9}
footer{margin-top:44px;padding-top:16px;border-top:1px solid var(--rule);font-size:11.5px;color:var(--faint);line-height:1.7}
</style>
</head>
<body>

<div class="demo"><b>데모 화면입니다.</b> 임상시험 정보는 ClinicalTrials.gov 공개 데이터를 조회하는 방식을 시연하며, 적합도 점수는 Gemini가 생성한 참고 추정치입니다. 실제 참여 적격 여부·연락은 각 실시기관의 절차를 따릅니다.</div>

<div class="wrap">

<header>
  <div class="brand"><h1>TrialMatch</h1><span class="tag">치료 선택지가 소진된 환자를 위한 임상시험 연결 검색</span><span class="eng off" id="engBadge" onclick="setBackend()" title="클릭: 라이브 백엔드(Colab) URL 설정" style="cursor:pointer">엔진 확인 중…</span></div>
</header>

<div class="irb">
  <b>연결 범위(윤리·규제 경계)</b> — 본 플랫폼은 <b>공개 등록되어 현재 '모집 중'인 임상시험</b>만 조회합니다(중재연구는 기관생명윤리위원회(IRB)/규제기관 승인 하에 등록·수행됩니다). 본 도구는 <b>환자를 직접 모집·중개하지 않으며</b>, 표시되는 연락처는 각 시험의 공식 등록 연락처입니다. 최종 참여 결정과 적격 판정은 <b>담당 의료진과 실시기관</b>이 수행합니다. 개인 식별정보는 입력하지 마세요.
</div>

<div class="searchcard">
  <label>환자 상황을 자유롭게 적어주세요 (진단명, 이전 치료, 바이오마커, 지역 등 — 식별정보 제외)</label>
  <textarea id="q" placeholder="예) 전신성 경화증에 동반된 폐동맥고혈압입니다. 3제 병용요법에도 6분 보행거리가 늘지 않고 WHO 기능분류 III입니다. 국내에서 참여 가능한 시험을 찾고 있어요."></textarea>
  <div class="chips" id="chips"></div>
  <div class="searchrow">
    <button class="go" id="btnSearch" onclick="search()">임상시험 검색</button>
    <label class="chk"><input type="checkbox" id="useDemo"> 표본 데이터로 미리보기 (실검색 안 함)</label>
    <span id="msg" style="font-size:12.5px;color:var(--muted)"></span>
  </div>
</div>

<div class="qbar" id="qbar" style="display:none">
  <div><div class="qsum" id="qsum"></div><div class="qterms" id="qterms"></div></div>
  <div class="srcbadge" id="srcbadge"></div>
</div>

<div class="split">
  <div id="resultsCol"></div>
  <div class="viewer" id="viewer">
    <div class="vempty">시험을 클릭하면 선정기준·기관·연락처와<br>담당 의사 전달용 문의 초안이 여기에 표시됩니다</div>
  </div>
</div>

<footer>
  본 화면은 창업 프로그램 제출용 프로토타입입니다. 임상시험 데이터는 ClinicalTrials.gov API를 통해 조회하는 방식을 가정하며(네트워크 미도달 시 '표본' 데이터로 대체 표시), 적합도 점수는 Gemini가 선정기준과 대조해 생성한 참고 추정치로 실제 적격성을 보장하지 않습니다. 본 도구는 진단·치료를 판정하지 않고 환자를 모집하지 않으며, 정보 제공과 담당 의료진 상담 보조만을 목적으로 합니다.
</footer>
</div>

<script>
const EXAMPLES=[
  "전신성 경화증에 동반된 폐동맥고혈압입니다. 3제 병용에도 6분 보행거리가 늘지 않고 WHO 기능분류 III입니다. 국내 시험을 찾고 있어요.",
  "희귀 고형암 환자입니다. 표준치료와 표적치료까지 모두 진행했고 ECOG 1, 특정 바이오마커 양성입니다.",
  "HER2 저발현 전이성 유방암, 이전 항암 2차 이상 진행했습니다. 새로운 ADC 시험이 있는지 궁금합니다."
];
// 백엔드 미연결/표본 미리보기용 임베디드 결과(구조는 실데이터와 동일, 명확히 표본 표시)
const SAMPLE={
 query:{summary_ko:"(표본) 결합조직질환 관련 폐동맥고혈압, 병용요법 반응 불충분 환자",condition_en:"systemic sclerosis pulmonary arterial hypertension",other_terms:"combination therapy",location_en:"Korea"},
 source:"sample",
 results:[
  {nctId:"NCT00000000",title:"[표본] 결합조직질환 관련 폐동맥고혈압 대상 신규 경구제 3상",status:"RECRUITING",phases:["PHASE3"],
   conditions:["Pulmonary Arterial Hypertension","Systemic Sclerosis"],interventions:["Investigational oral agent"],
   eligibilityText:"Inclusion: adults with CTD-associated PAH, WHO FC II-III, 6MWD 150-450m, on stable background therapy >=3 months.\nExclusion: PVOD, severe hepatic impairment.",
   sex:"ALL",minAge:"19 Years",maxAge:"80 Years",
   locations:[{facility:"[표본] 서울 소재 대학병원",city:"Seoul",country:"Korea, Republic of",status:"RECRUITING"}],
   contacts:[{name:"[표본] 임상시험 코디네이터",phone:"02-000-0000",email:"trial@example.org"}],
   url:"https://clinicaltrials.gov/",
   matchPct:88,summary:"질환·기능분류·병용치료 조건이 선정기준과 대체로 부합합니다.",note:"6MWD 범위와 배경치료 유지 기간 확인 필요.",
   crit:[["y","결합조직질환 관련 폐동맥고혈압"],["y","WHO 기능분류 II–III"],["q","6분 보행거리 150–450m — 수치 확인 필요"],["y","배경치료 3개월 이상 유지"],["q","우측 심도자 검사 시점 확인"]]},
  {nctId:"NCT00000002",title:"[표본] 폐동맥고혈압 환자 레지스트리 기반 관찰연구",status:"NOT_YET_RECRUITING",phases:[],
   conditions:["Pulmonary Hypertension"],interventions:[],
   eligibilityText:"Inclusion: confirmed pulmonary hypertension, age >=19. Observational, no additional intervention.",
   sex:"ALL",minAge:"19 Years",maxAge:"N/A",
   locations:[{facility:"[표본] 국내 다기관",city:"Seoul",country:"Korea, Republic of",status:"NOT_YET_RECRUITING"}],
   contacts:[{name:"[표본] 연구간호사",phone:"02-000-0002",email:"registry@example.org"}],
   url:"https://clinicaltrials.gov/",
   matchPct:61,summary:"관찰연구로 참여 부담은 낮으나 직접적 치료 효과는 기대하기 어렵습니다.",note:"중재가 없어 치료 목적과는 거리가 있음.",
   crit:[["y","폐고혈압 확진"],["y","연령 19세 이상"],["q","정기 추적 가능 여부 확인 필요"]]}
 ]
};
let LAST=null,active=null;

// --- 백엔드 URL 설정 (GitHub Pages 등 정적 호스팅에서도 Colab 백엔드에 연결) -----
let API_BASE=(localStorage.getItem('trialmatch_api')||'').replace(/\/+$/,'');
const api=p=>API_BASE?API_BASE+'/'+p:p;
function setBackend(){
  const cur=localStorage.getItem('trialmatch_api')||'';
  const u=prompt('라이브 검색을 쓰려면 실행 중인 백엔드 URL을 입력하세요.\n(Colab의 https://….trycloudflare.com 또는 http://localhost:8001)\n비우면 표본(데모) 모드로 동작합니다.',cur);
  if(u===null)return;
  API_BASE=u.trim().replace(/\/+$/,'');
  localStorage.setItem('trialmatch_api',API_BASE);
  document.getElementById('msg').textContent=API_BASE?('백엔드 설정됨 · '+API_BASE):'표본 모드로 전환됨';
  checkEngine();
}

const STMAP={RECRUITING:["모집중","rec"],NOT_YET_RECRUITING:["모집예정","soon"],ACTIVE_NOT_RECRUITING:["모집종료·진행중","soon"]};
const PHMAP={PHASE1:"1상",PHASE2:"2상",PHASE3:"3상",PHASE4:"4상",EARLY_PHASE1:"초기 1상",NA:"해당없음"};
function phaseLabel(ph){return (ph&&ph.length)?ph.map(x=>PHMAP[x]||x).join("·"):"단계 미표기";}
function normCrit(c){return (c||[]).map(x=>Array.isArray(x)?x:[x.status,x.text]);}

async function checkEngine(){
  const b=document.getElementById('engBadge');
  try{const r=await fetch(api('api/health'));const h=await r.json();
    if(h.key_present){b.textContent='Gemini · '+h.model;b.classList.remove('off');}
    else{b.textContent='엔진 연결됨 · API 키 없음';b.classList.add('off');}
  }catch(e){b.textContent='백엔드 없음 (표본 모드)';b.classList.add('off');document.getElementById('useDemo').checked=true;}
}
function renderChips(){document.getElementById('chips').innerHTML=EXAMPLES.map((e,i)=>
  `<span class="chip" onclick="useExample(${i})">${e.slice(0,26)}…</span>`).join('');}
function useExample(i){document.getElementById('q').value=EXAMPLES[i];}

async function search(){
  const text=document.getElementById('q').value.trim();
  const b=document.getElementById('btnSearch'),m=document.getElementById('msg');
  if(!text){m.textContent="환자 상황을 먼저 입력하세요.";return;}
  if(document.getElementById('useDemo').checked){show(SAMPLE);return;}
  b.disabled=true;m.innerHTML='<span class="spin"></span>Gemini로 검색어를 해석하고 모집중 임상시험을 대조하는 중… (10~30초)';
  try{
    const r=await fetch(api('api/search'),{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({text})});
    if(!r.ok){const j=await r.json().catch(()=>({}));throw new Error(j.error||('HTTP '+r.status));}
    const data=await r.json();b.disabled=false;m.textContent="";show(data);
  }catch(e){b.disabled=false;m.innerHTML='⚠ 백엔드/네트워크 실패 — 표본 데이터로 표시합니다. ('+e.message+')';show(SAMPLE);}
}

function show(data){
  LAST=data;active=null;
  const q=data.query||{};
  document.getElementById('qbar').style.display='flex';
  document.getElementById('qsum').innerHTML='<b>해석된 검색:</b> '+(q.summary_ko||'');
  document.getElementById('qterms').textContent=[q.condition_en,q.other_terms,q.location_en].filter(Boolean).join(' · ');
  const sb=document.getElementById('srcbadge');
  if(data.source==='clinicaltrials.gov'){sb.className='srcbadge live';sb.textContent='실데이터 · ClinicalTrials.gov';}
  else{sb.className='srcbadge sample';sb.textContent='표본 데이터 (실 API 미도달/미리보기)';}
  renderResults();
  document.getElementById('viewer').innerHTML='<div class="vempty">시험을 클릭하면 선정기준·기관·연락처와<br>담당 의사 전달용 문의 초안이 여기에 표시됩니다</div>';
  window.scrollTo({top:document.getElementById('qbar').offsetTop-20,behavior:'smooth'});
}

function renderResults(){
  const res=(LAST.results||[]);
  const col=document.getElementById('resultsCol');
  if(!res.length){col.innerHTML='<div class="empty">조건에 맞는 모집중 임상시험을 찾지 못했습니다.<br>검색어를 더 일반적인 진단명으로 바꾸거나 지역 조건을 빼보세요.</div>';return;}
  col.innerHTML=res.map((t,i)=>{
    const st=STMAP[t.status]||[t.status||'상태미상','soon'];
    const pct=t.matchPct;
    const conds=(t.conditions||[]).slice(0,4).map(c=>`<span class="cond">${c}</span>`).join('');
    return `<div class="tcard${active===i?' act':''}" onclick="showTrial(${i})">
      <div class="thead">
        <span class="tstatus ${st[1]}">${st[0]}</span>
        <div class="ttl">${t.title}</div>
        <div class="meta">${t.nctId||''} · ${phaseLabel(t.phases)}${t.locations&&t.locations[0]?' · '+(t.locations[0].country||''):''}</div>
        <div class="conds">${conds}</div>
        ${t.summary?`<div class="sum">${t.summary}</div>`:''}
        <div class="matchbar"><div class="matchtrack"><div class="matchfill" style="width:${pct||0}%"></div></div>
          <span class="matchpct${pct==null?' none':''}">${pct==null?'점수없음':'적합도 '+pct}</span></div>
      </div>
      <div class="foot"><span>선정기준 ${normCrit(t.crit).filter(c=>c[0]==='y').length}/${normCrit(t.crit).length}항목 부합 · 클릭하면 상세</span>
        <span>${(t.contacts&&t.contacts.length)?'연락처 있음':'연락처 미표기'}</span></div>
    </div>`;
  }).join('');
}

function inquiryDraft(t){
  const patient=document.getElementById('q').value.trim();
  const st=(STMAP[t.status]||['모집'])[0];
  return `[임상시험 참여 문의 초안 — 담당 의료진 전달용]

안녕하세요. 아래 임상시험 참여 가능성을 문의드립니다.
· 시험명: ${t.title}
· 등록번호: ${t.nctId} (ClinicalTrials.gov)
· 상태: ${st}
· 등록정보: ${t.url||'ClinicalTrials.gov'}

환자 상황(요약): ${patient||'(입력한 상황)'}

선정/제외 기준 충족 여부와 다음 절차를 안내 부탁드립니다.
※ 본 문의는 참고용 초안이며, 최종 적격 여부는 실시기관의 선별검사로 확정됩니다.`;
}

function showTrial(i){
  active=i;const t=LAST.results[i];
  const lab={y:"부합",n:"불충족",q:"확인 필요"};
  const crit=normCrit(t.crit);
  const critHtml=crit.length?`<ul class="crit">${crit.map(c=>`<li class="${c[0]}">${c[1]} <span style="color:var(--faint);font-size:11px">(${lab[c[0]]||'확인 필요'})</span></li>`).join('')}</ul>`:'<div class="note">선정기준 요약이 제공되지 않았습니다. 원문을 확인하세요.</div>';
  const locs=(t.locations||[]);
  const locHtml=locs.length?locs.map(l=>`<div class="loc">${l.facility||'(기관명 미표기)'}<div class="lc">${[l.city,l.country].filter(Boolean).join(', ')}${l.status?' · '+l.status:''}</div></div>`).join(''):'<div class="note">등록된 실시기관 정보가 없습니다.</div>';
  const cons=(t.contacts||[]).filter(c=>c.name||c.phone||c.email);
  const conHtml=cons.length?cons.map(c=>`<div class="contact">${[c.name,c.phone,c.email].filter(Boolean).join(' · ')}</div>`).join(''):'<div class="note">공식 등록 연락처가 없습니다. 담당 의료진을 통해 실시기관에 문의하세요.</div>';
  const demo=[t.minAge,t.maxAge].filter(x=>x&&x!=='N/A').join(' ~ ');
  document.getElementById('viewer').innerHTML=`<div class="vbox">
    <div class="vhead"><div class="vt">${t.title}</div>
      <div class="vm">${t.nctId} · ${phaseLabel(t.phases)}${demo?' · '+demo:''}${t.sex&&t.sex!=='ALL'?' · '+t.sex:''}</div></div>
    <div class="vsec"><div class="vl">환자 기준 선정·제외 대조 ${t.matchPct!=null?'· 적합도 '+t.matchPct:''}</div>${critHtml}
      ${t.note?`<div class="note">${t.note}</div>`:''}</div>
    <div class="vsec"><div class="vl">원문 선정기준 (eligibility)</div>
      <div class="eltext">${(t.eligibilityText||'원문 미제공').replace(/</g,'&lt;')}</div></div>
    <div class="vsec"><div class="vl">실시기관</div>${locHtml}</div>
    <div class="vsec"><div class="vl">공식 등록 연락처</div>${conHtml}
      ${t.url?`<div class="btnrow"><a class="ext" href="${t.url}" target="_blank" rel="noopener">ClinicalTrials.gov 원문 보기 ↗</a></div>`:''}</div>
    <div class="vsec"><div class="vl">담당 의료진 전달용 문의 초안</div>
      <div class="inq" id="inqText">${inquiryDraft(t)}</div>
      <div class="btnrow"><button class="ghost" onclick="copyInq()">문의 초안 복사</button></div>
      <div class="note">본 플랫폼은 환자를 직접 모집·중개하지 않습니다. 위 초안을 담당 의료진과 상의해 활용하세요.</div></div>
  </div>`;
  renderResults();
}
function copyInq(){const el=document.getElementById('inqText');
  navigator.clipboard&&navigator.clipboard.writeText(el.textContent).then(()=>{
    const b=event.target;const o=b.textContent;b.textContent='복사됨 ✓';setTimeout(()=>b.textContent=o,1500);});}

renderChips();checkEngine();
</script>
</body>
</html>


## 3) Gemini API 키 입력
키 발급: <https://aistudio.google.com/app/apikey>  (왼쪽 🔑 보안 비밀에 `GEMINI_API_KEY` 저장도 가능)

In [ ]:
# 직접 입력 칸 + 저장 버튼. (연결 확인이 실패해도 키만 맞으면 4번 셀로 진행 가능)
import os
os.environ.setdefault('GEMINI_MODEL', 'gemini-2.5-flash')

_pre = ''
try:
    from google.colab import userdata
    _pre = userdata.get('GEMINI_API_KEY') or ''
except Exception:
    pass

def _verify():
    try:
        from google import genai
        client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY', ''))
        client.models.generate_content(model=os.environ['GEMINI_MODEL'], contents='ping')
        print('✅ Gemini 연결 OK · 모델:', os.environ['GEMINI_MODEL'])
    except Exception as e:
        print('⚠️ 연결 확인만 실패(키가 맞아도 날 수 있음):', e)
        print('   → 키를 칸에 제대로 넣었다면 4번 "서버 실행" 셀로 그냥 넘어가도 됩니다.')

try:
    import ipywidgets as w
    from IPython.display import display
    _key = w.Text(value=_pre, description='API Key', placeholder='여기에 키를 붙여넣기',
                  layout=w.Layout(width='620px'), style={'description_width': '70px'})
    _model = w.Text(value=os.environ['GEMINI_MODEL'], description='Model',
                    layout=w.Layout(width='380px'), style={'description_width': '70px'})
    _btn = w.Button(description='저장하고 확인', button_style='success')
    _out = w.Output()
    if _pre:
        os.environ['GEMINI_API_KEY'] = _pre
    def _save(_):
        with _out:
            _out.clear_output()
            os.environ['GEMINI_API_KEY'] = _key.value.strip()
            os.environ['GEMINI_MODEL'] = _model.value.strip() or 'gemini-2.5-flash'
            if not os.environ['GEMINI_API_KEY']:
                print('⚠️ 키 칸이 비어 있어요. 붙여넣고 다시 누르세요.'); return
            print('저장됨. 확인 중…'); _verify()
    _btn.on_click(_save)
    display(w.VBox([_key, w.HBox([_model, _btn]), _out]))
    print('↑ 칸에 키를 붙여넣고 [저장하고 확인]을 누르세요.')
except Exception:
    os.environ['GEMINI_API_KEY'] = input('Gemini API Key 를 붙여넣고 Enter: ').strip()
    _verify()

# ClinicalTrials.gov 연결 확인 (Colab 은 외부망이 열려 있어 정상 조회됨)
import urllib.request, json
try:
    u = "https://clinicaltrials.gov/api/v2/studies?query.cond=pulmonary%20arterial%20hypertension&filter.overallStatus=RECRUITING&pageSize=1&format=json"
    d = json.loads(urllib.request.urlopen(u, timeout=20).read())
    print('✅ ClinicalTrials.gov 연결 OK · 예시 조회 건수:', len(d.get('studies', [])))
except Exception as e:
    print('⚠️ CT.gov 조회 실패(표본 데이터로 대체됨):', e)


## 4) 서버 실행 → 공개 URL

출력되는 `https://….trycloudflare.com` 주소를 **새 탭에서 열면** 이 앱(TrialMatch) 검색창이 뜬다.
이 셀은 서버라 **계속 실행 상태로 둔다**(중지: ⏹️).

In [ ]:
import sys
sys.path.insert(0, 'trialmatch')
sys.modules.pop('trialmatch_app', None)
from trialmatch_app import create_app
from flask_cloudflared import run_with_cloudflared

app = create_app()
print('>>> 실행 중인 앱: TrialMatch (trialmatch_app, port 8001)')
run_with_cloudflared(app)
app.run(port=8001)


### (대안) cloudflared 가 안 될 때 — Colab 내장 프록시

In [ ]:
import sys, threading
sys.path.insert(0, 'trialmatch')
sys.modules.pop('trialmatch_app', None)
from trialmatch_app import create_app
app = create_app()
threading.Thread(target=lambda: app.run(port=8001, use_reloader=False), daemon=True).start()
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8001)
